# Clearvale: run length of every BOLD run from Flywheel metadata

Reads metadata only (nothing is downloaded or changed). For every `func-bold_*` acquisition (SBRef excluded) it takes the DICOM zip's
`zip_member_count` -- one DICOM per volume on this Siemens multiband protocol, so it is the number of volumes -- plus the
`RepetitionTime` from the DICOM header and the scan date. Checked on sub-090: 1240 (loc) and 770 (WM) members = the BOLD
NIfTI's 4th dimension.

Writes `sourcedata/run_lengths.csv`; `code/make_events.py` uses it for a run's length (where `EndFixation` ends) when the
run's preprocessed BOLD is not downloaded. Uses the saved Flywheel CLI login (`fw login`), like notebook 03.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import flywheel

# the Clearvale BIDS folder: $CV_BIDS, else <repo>/data/cv-bids (run this notebook from examples/clearvale/notebooks)
OUT = Path(os.environ.get("CV_BIDS", Path.cwd().resolve().parents[2] / "data" / "cv-bids"))
PROJECT = "mbanich/CV"
SUBJECTS = None          # None = every subject; or a list of labels

fw = flywheel.Client()
project = fw.lookup(PROJECT)
print("project:", project.label)

## Inventory

In [ ]:
rows = []
subjects = sorted((s for s in project.subjects() if SUBJECTS is None or s.label in SUBJECTS), key=lambda s: s.label)
for i, subj in enumerate(subjects):
    for ses in subj.sessions():
        for acq in ses.acquisitions():
            if not acq.label.startswith("func-bold") or acq.label.endswith("SBRef"):
                continue
            acq = acq.reload()
            dicoms = [f for f in acq.files or [] if f.type == "dicom"]
            niftis = [f for f in acq.files or [] if f.type == "nifti"]
            if not dicoms:
                rows.append({"subject": subj.label, "session": ses.label, "acquisition": acq.label,
                             "n_dicom_files": 0, "n_nifti_files": len(niftis)})
                continue
            for f in dicoms:
                info = f.info or {}
                tr = info.get("RepetitionTime")
                rows.append({"subject": subj.label, "session": ses.label, "acquisition": acq.label,
                             "date": acq.timestamp.date().isoformat() if acq.timestamp else None,
                             "dicom_file": f.name, "n_dicom_files": len(dicoms), "n_nifti_files": len(niftis),
                             "volumes": f.get("zip_member_count"),
                             "tr_s": tr / 1000 if tr and tr > 20 else tr,   # DICOM header is in ms
                             "dicom_mb": round(f.size / 1e6, 1)})
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(subjects)} subjects")
runs = pd.DataFrame(rows)
m = runs["acquisition"].str.extract(r"task-(?P<task>[A-Za-z]+)_dir-(?P<dir>[A-Za-z]+)_run-(?P<run>\d+)")
runs = pd.concat([runs, m], axis=1)
runs["length_s"] = runs["volumes"] * runs["tr_s"]
runs["ignored"] = runs["acquisition"].str.contains("ignore", case=False)   # aborted runs, e.g. *_ignore-BIDS
runs.to_csv(OUT / "sourcedata" / "run_lengths.csv", index=False)
print(f"{len(runs)} BOLD runs, {runs['subject'].nunique()} subjects -> sourcedata/run_lengths.csv")

## Does the run length change during the study?

In [ ]:
runs["task_group"] = runs["task"].where(runs["task"] == "loc", "WM")
kept = runs[~runs["ignored"]]
display(kept.groupby(["task_group", "volumes", "tr_s"], dropna=False).size().rename("runs").to_frame())

# per task, the first and last scan date of each run length
display(kept.groupby(["task_group", "volumes"]).agg(first=("date", "min"), last=("date", "max"),
                                                     subjects=("subject", "nunique")))

In [ ]:
# runs that are not the session's usual length, and acquisitions with 0 or 2+ DICOM files (repeats / missing)
usual = kept.groupby("task_group")["volumes"].agg(lambda v: v.mode().iloc[0])
odd = kept[kept["volumes"] != runs["task_group"].map(usual)]
print("runs with an unusual volume count:"); display(odd[["subject", "session", "acquisition", "date", "volumes", "length_s"]])
print("acquisitions without exactly one DICOM zip:")
display(runs[runs["n_dicom_files"] != 1][["subject", "session", "acquisition", "n_dicom_files", "n_nifti_files"]].drop_duplicates())